# Dia 09 — Automatizar a coleta de resultados de busca

**🃏 Cartas:** 9 · 30 · 51 &nbsp;·&nbsp; **📘 Original:** *Day 9. Automate the collection of search results*
&nbsp;·&nbsp; [repositório](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias) &nbsp;·&nbsp; [curso original do cipher387](https://github.com/cipher387/python-for-OSINT-21-days)

---

## 📖 Capítulo 9 — Rastros na busca

Sexta-feira, 8h20. Na parede do NFA, um fio vermelho agora liga três nomes de titulares a um único
post-it: **painel_dump.sql**. A Iara fica um tempo olhando para ele, depois vira a cadeira.

— Sabemos quem perdeu e quem recebeu. Agora: **o que o mundo já sabe sobre essa loja?** Golpe assim
não nasce do nada. Alguém reclamou num fórum, alguém postou o CNPJ falso num grupo, alguém usou o
mesmo apelido em outro golpe.

Você abre o navegador, mas ela fecha a aba com um gesto.

— Buscar à mão a gente faz uma vez. Investigação séria repete a mesma busca todo dia, em vários
lugares, e guarda **o que viu e quando viu**. Para isso servem os scripts. Mas cuidado: raspar o
Google é pedir para ser bloqueado e ainda quebra os termos de uso. Você vai montar as buscas, e quem
clica é você.

Ela escreve no quadro: **Querido Diário**.

— E tem uma fonte que quase ninguém lembra: o Diário Oficial. Prefeitura publica alerta ao
consumidor, contrato, sanção, empresa impedida de licitar. Se esse CNPJ falso ou esse sobrenome
aparecer lá, quero saber antes da Delegada Rita perguntar.

In [ ]:
#@title ⚙️ Preparar o ambiente (rode esta célula primeiro)
# No Colab: baixa o repositório e entra na pasta do dia.
# No Jupyter local: o notebook já está na pasta do dia, então não faz nada.
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias"
if "google.colab" in sys.modules:
    destino = "/content/python-para-osint-21-dias"
    if not os.path.isdir(destino):
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    os.chdir(f"{destino}/dia-09-resultados-de-busca")

PASTA_NB = str(Path.cwd())          # usada pelas células para voltar sempre à pasta certa
print("📂 Pasta do dia:", PASTA_NB)
print("🐍 Python", sys.version.split()[0])

## 🎯 Objetivos

- Rodar a coleta de resultados de busca do original e entender por que ela precisou de atualização.
- Usar a versão 2026 com o pacote `ddgs` (`DDGS().text()`).
- Consultar os **Diários Oficiais municipais** pela API do **Querido Diário**.
- Salvar resultados de busca em CSV com a data e a hora da coleta.
- Dominar os **operadores de busca** (dorks) e gerar links clicáveis para vários buscadores.

## 🧠 Conceito em 2 minutos

Buscadores entendem **operadores** que refinam a busca. Saber combiná-los é uma das habilidades mais
valiosas em OSINT:

| Operador | Exemplo | Significa |
|---|---|---|
| `"…"` | `"kau.ofertas"` | exatamente esta sequência |
| `site:` | `site:gov.br` | só páginas deste domínio (e subdomínios) |
| `filetype:` | `filetype:pdf` | só arquivos deste tipo |
| `intext:` / `intitle:` | `intitle:"golpe"` | o termo no texto / no título |
| `-` | `-site:exemplo.com` | exclui |
| `OR` | `golpe OR fraude` | um ou outro |

Existem dois jeitos de automatizar: **(1)** bibliotecas que conversam com o buscador por você (como a
`ddgs`, usada pelo autor), práticas mas frágeis, porque o buscador muda e pode bloquear; e **(2)**
**APIs oficiais** de fontes específicas, como o **Querido Diário**, que existem justamente para serem
consultadas por programas. Quando houver API, prefira a API.

## 🥋 Nível 1 — Exercício original (cipher387)

| Arquivo | O que faz |
|---|---|
| [`original/ddg_search.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-09-resultados-de-busca/original/ddg_search.py) | Busca "osint" no DuckDuckGo com `ddg()` e imprime os resultados |
| [`original/ddg_search_to_csv.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-09-resultados-de-busca/original/ddg_search_to_csv.py) | Salva título, resumo e link em `search_results.csv` |
| [`original/ddg_search_download_pdf.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-09-resultados-de-busca/original/ddg_search_download_pdf.py) | Busca com `filetype:pdf` e baixa os arquivos (`download=True`) |

> ⚠️ **Os três originais não rodam mais.** A função `ddg()` do pacote `duckduckgo_search` foi removida,
> e o projeto passou a se chamar **`ddgs`**. Os originais ficam intactos (é a regra); ao lado, estão as
> versões atualizadas:

| Versão 2026 | O que mudou |
|---|---|
| [`original/2026_ddg_search.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-09-resultados-de-busca/original/2026_ddg_search.py) | `DDGS().text(..., timelimit="y", max_results=10)` |
| [`original/2026_ddg_search_to_csv.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-09-resultados-de-busca/original/2026_ddg_search_to_csv.py) | idem + `newline=""` e `encoding="utf-8"` no CSV |
| [`original/2026_ddg_search_download_pdf.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-09-resultados-de-busca/original/2026_ddg_search_download_pdf.py) | busca com `ddgs` e download de cada PDF com `urllib` |

```bash
pip install ddgs
cd dia-09-resultados-de-busca/original
python 2026_ddg_search.py
python 2026_ddg_search_to_csv.py
```

> 💡 Bibliotecas que conversam com buscadores mudam com frequência. Se o `ddgs` reclamar, atualize
> (`pip install -U ddgs`) e confira o README do projeto. O autor também lista alternativas: `ddgr`
> (DuckDuckGo no terminal), `Search-Engines-Scraper` e outras. E a dica dele vale para quase todo
> pacote: digite `--help` (ou `-h`) depois do nome de um comando para ver como usá-lo.

In [ ]:
#@title 📦 Instalar as bibliotecas do exercício original
%pip install -q duckduckgo_search

### `original/2026_ddg_search.py` — Buscar no DuckDuckGo a partir do Python (versão 2026)

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Buscar no DuckDuckGo a partir do Python (versão 2026)
# ATUALIZAÇÃO feita para este repositório, BASEADA no original de cipher387
# (Cyb_detective): https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_9/ddg_search.py
# Licença: MIT © 2023 Cyb_detective (original). Por que existe: a função ddg()
# do pacote duckduckgo_search foi removida; o pacote hoje se chama "ddgs" e usa
# a classe DDGS. O original continua intacto em original/ddg_search.py.
# Requer: pip install ddgs
# ──────────────────────────────────────────────────────────────────────────

from ddgs import DDGS

keywords = 'osint'

# Antes: ddg(keywords, region='us-en', safesearch='Off', time='y')
# Agora: o parâmetro "time" virou "timelimit", safesearch é minúsculo ("off")
# e max_results diz quantos resultados queremos.
results = DDGS().text(keywords, region="us-en", safesearch="off", timelimit="y", max_results=10)

# Continua sendo uma lista de dicionários com as chaves title, href e body.
print(results)

### `original/2026_ddg_search_download_pdf.py` — Buscar PDFs no DuckDuckGo e baixar os arquivos (versão 2026)

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Buscar PDFs no DuckDuckGo e baixar os arquivos (versão 2026)
# ATUALIZAÇÃO feita para este repositório, BASEADA no original de cipher387
# (Cyb_detective): https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_9/ddg_search_download_pdf.py
# Licença: MIT © 2023 Cyb_detective (original). A opção download=True deixou de
# existir na API Python; aqui fazemos a busca com DDGS().text() e baixamos cada
# PDF com urllib (biblioteca padrão). O original continua intacto ao lado.
# Requer: pip install ddgs
# ──────────────────────────────────────────────────────────────────────────

import urllib.request
from pathlib import Path

from ddgs import DDGS

keywords = 'open source intelligence filetype:pdf'
max_results = 5                      # quantos arquivos, no máximo, vamos baixar

results = DDGS().text(keywords, region="us-en", safesearch="off", timelimit="y", max_results=max_results)

pasta = Path("pdfs_baixados")
pasta.mkdir(exist_ok=True)

for numero, resultado in enumerate(results, start=1):
    link = resultado["href"]
    if not link.lower().split("?")[0].endswith(".pdf"):
        continue                     # o buscador às vezes devolve páginas, não PDFs
    destino = pasta / f"{numero:02d}_{Path(link.split('?')[0]).name}"
    try:
        pedido = urllib.request.Request(link, headers={"User-Agent": "python-para-osint-21-dias/estudo"})
        with urllib.request.urlopen(pedido, timeout=30) as resposta:
            destino.write_bytes(resposta.read())
        print("baixado:", destino)
    except Exception as erro:        # um link quebrado não deve parar os outros
        print("falhou:", link, erro)

### `original/2026_ddg_search_to_csv.py` — Salvar resultados de busca do DuckDuckGo num CSV (versão 2026)

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Salvar resultados de busca do DuckDuckGo num CSV (versão 2026)
# ATUALIZAÇÃO feita para este repositório, BASEADA no original de cipher387
# (Cyb_detective): https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_9/ddg_search_to_csv.py
# Licença: MIT © 2023 Cyb_detective (original). Troca ddg() (removida) por
# DDGS().text() do pacote "ddgs". O original continua em original/ddg_search_to_csv.py.
# Requer: pip install ddgs
# ──────────────────────────────────────────────────────────────────────────

from ddgs import DDGS
import csv

# Pequenas melhorias em relação ao original: newline="" e encoding="utf-8"
# (no Windows, sem isso, aparecem linhas vazias e acentos quebrados).
csv_file = open('search_results.csv', 'w', newline='', encoding='utf-8')
writer = csv.writer(csv_file, delimiter=';')

keywords = 'osint'
results = DDGS().text(keywords, region="us-en", safesearch="off", timelimit="y", max_results=10)

for x in range(len(results)):
    row = [results[x]["title"], results[x]["body"], results[x]["href"]]
    writer.writerow(row)

csv_file.close()

### `original/ddg_search.py` — Buscar no DuckDuckGo a partir do Python

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Buscar no DuckDuckGo a partir do Python
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_9/ddg_search.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Requer (na época): pip install duckduckgo_search
# ⚠️ Em 2026 este código NÃO roda mais: a função ddg() foi removida do pacote.
#    Use original/2026_ddg_search.py (versão atualizada com o pacote ddgs).
# ──────────────────────────────────────────────────────────────────────────

# 💬 ddg era uma função "atalho" do pacote duckduckgo_search (versões de 2022/2023).
from duckduckgo_search import ddg

# 💬 O termo de busca.
keywords = 'osint'
# 💬 region='us-en' = resultados dos EUA em inglês; safesearch='Off' = sem filtro;
# 💬 time='y' = só páginas do último ano (year).
results = ddg(keywords, region='us-en', safesearch='Off', time='y')
# 💬 results era uma lista de dicionários com title, href e body.
print(results)

### `original/ddg_search_download_pdf.py` — Buscar PDFs no DuckDuckGo e baixar os arquivos encontrados

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Buscar PDFs no DuckDuckGo e baixar os arquivos encontrados
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_9/ddg_search_download_pdf.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Requer (na época): pip install duckduckgo_search
# ⚠️ Em 2026 este código NÃO roda mais (ddg() e a opção download=True foram removidas).
#    Veja original/2026_ddg_search_download_pdf.py.
# ──────────────────────────────────────────────────────────────────────────

from duckduckgo_search import ddg

# 💬 filetype:pdf é um operador de busca: só traz arquivos PDF.
keywords = 'open source intelligence filetype:pdf'
# 💬 download=True mandava o pacote baixar cada arquivo encontrado para uma pasta local.
# 💬 O autor lembra que max_results controla quantos arquivos são baixados.
results = ddg(keywords, region='us-en', safesearch='Off', time='y', download=True)

### `original/ddg_search_to_csv.py` — Salvar resultados de busca do DuckDuckGo num CSV

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Salvar resultados de busca do DuckDuckGo num CSV
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_9/ddg_search_to_csv.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Requer (na época): pip install duckduckgo_search
# ⚠️ Em 2026 este código NÃO roda mais (ddg() foi removida).
#    Use original/2026_ddg_search_to_csv.py.
# ──────────────────────────────────────────────────────────────────────────

from duckduckgo_search import ddg
import csv

# 💬 Mesmo padrão do Dia 7: arquivo aberto para escrita e separador ";".
csv_file = open('search_results.csv', 'w')
writer = csv.writer(csv_file, delimiter =';')


keywords = 'osint'
results = ddg(keywords, region='us-en', safesearch='Off', time='y')

# 💬 Uma linha do CSV por resultado: título, resumo (body) e link (href).
for x in range(len(results)):
    row = [results[x]["title"],results[x]["body"],results[x]["href"]]
    writer.writerow(row)

csv_file.close()

## 🇧🇷 Nível 2 — Camada Brasil

Tudo com a **biblioteca padrão**. Amostra offline do Querido Diário em `dados/`.

| Arquivo | O que faz |
|---|---|
| [`br/01_querido_diario.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-09-resultados-de-busca/br/01_querido_diario.py) | Busca um termo nos Diários Oficiais de um município (padrão: "PIX" em BH, código IBGE 3106200) |
| [`br/02_querido_diario_csv.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-09-resultados-de-busca/br/02_querido_diario_csv.py) | Vários termos × vários municípios, com pausa, sem duplicatas, num CSV com data da coleta |
| [`br/03_dorks_br.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-09-resultados-de-busca/br/03_dorks_br.py) | Monta dorks (treino em `gov.br` e do caso) para Google, DuckDuckGo e Bing e grava `saidas/dorks.html` |

```bash
cd dia-09-resultados-de-busca
python br/01_querido_diario.py
OSINT_OFFLINE=1 python br/02_querido_diario_csv.py     # força o modo offline
python br/03_dorks_br.py                               # depois abra saidas/dorks.html
```

**O que você deve ver no caso:** o domínio, o apelido e o CNPJ do caso são **fictícios**, então as
dorks do caso devem voltar vazias (e isso é bom: prova que você não está mexendo em gente real). As
dorks de treino, em `gov.br`, mostram os operadores funcionando. A amostra do Querido Diário tem
trechos **inventados**, marcados como "(TRECHO FICTÍCIO)": com internet, você vê publicações reais.

> 🧭 **Querido Diário, na prática:** os campos que importam são `total_gazettes` (quantos diários
> batem com a busca) e `gazettes`, uma lista com `date`, `territory_name`, `url` (o PDF original) e
> `excerpts` (trechos onde o termo aparece). Nem todo município está coberto; confira a lista no site.

> 💡 Os campos com formulário à direita (`#@param`) podem ser editados sem mexer no código. Para testar sem internet, rode antes: `os.environ['OSINT_OFFLINE'] = '1'`.

### `br/01_querido_diario.py` — Buscar palavras nos Diários Oficiais dos municípios com a API do Querido Diário.

In [ ]:
%cd -q {PASTA_NB}
"""
Buscar palavras nos Diários Oficiais dos municípios com a API do Querido Diário.

Carta 30 (rank B) — Jutsu do Diário Oficial

O que você aprende:
  • a montar uma URL com vários parâmetros (querystring, territory_ids, datas, tamanhos);
  • a ler uma resposta paginada: total_gazettes (quantos existem) × gazettes (quantos vieram);
  • a tratar a resposta com .get e, se a estrutura mudar, mostrar as chaves recebidas;
  • que Diário Oficial é fonte aberta OFICIAL: contratos, nomeações, licenças, sanções.

Como rodar (dentro da pasta dia-09-resultados-de-busca):
    python br/01_querido_diario.py
    OSINT_OFFLINE=1 python br/01_querido_diario.py
"""

import json
import os
import urllib.parse
import urllib.request
from pathlib import Path

# ── Parâmetros ────────────────────────────────────────────────────────────
TERMO = "PIX"  #@param {type:"string"}
CODIGO_IBGE = "3106200"  #@param {type:"string"}
DESDE = "2025-10-01"  #@param {type:"date"}
ATE = "2025-12-31"  #@param {type:"date"}
QUANTOS = 5  #@param {type:"integer"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
OFFLINE = os.environ.get("OSINT_OFFLINE") == "1"


def buscar_json(url, amostra):
    """Baixa um JSON da internet; se falhar (ou OFFLINE=1), lê a amostra local."""
    if not OFFLINE:
        try:
            pedido = urllib.request.Request(url, headers={"User-Agent": "python-para-osint-21-dias/estudo"})
            with urllib.request.urlopen(pedido, timeout=30) as resposta:
                return json.load(resposta)
        except Exception as erro:
            print(f"⚠️ MODO OFFLINE: a consulta falhou ({erro}). Usando a amostra local.")
    else:
        print("⚠️ MODO OFFLINE: usando a amostra local.")
    return json.loads((PASTA_DIA / "dados" / amostra).read_text(encoding="utf-8"))


# ── 1. Montar a URL ───────────────────────────────────────────────────────
# 3106200 é o código IBGE de Belo Horizonte (o mesmo do Dia 7). Cada parâmetro:
#   querystring        → o que procurar no texto dos diários
#   territory_ids      → em qual município (código IBGE)
#   published_since/until → período (AAAA-MM-DD)
#   size               → quantos diários devolver nesta página
#   excerpt_size       → tamanho (em caracteres) de cada trecho com a palavra
#   number_of_excerpts → quantos trechos por diário
parametros = {
    "querystring": TERMO,
    "territory_ids": CODIGO_IBGE,
    "published_since": DESDE,
    "published_until": ATE,
    "size": QUANTOS,
    "excerpt_size": 250,
    "number_of_excerpts": 1,
}
url = "https://api.queridodiario.ok.org.br/gazettes?" + urllib.parse.urlencode(parametros)
print(f"🌐 GET {url}\n")

resposta = buscar_json(url, "querido_diario_bh.json")

# ── 2. A estrutura veio como esperado? ────────────────────────────────────
# Esperamos {"total_gazettes": N, "gazettes": [...]}. APIs mudam; se mudar, avisamos.
if not isinstance(resposta, dict) or "gazettes" not in resposta:
    chaves = list(resposta) if isinstance(resposta, dict) else type(resposta).__name__
    raise SystemExit(f"🤔 Estrutura diferente da esperada. Chaves recebidas: {chaves}")

diarios = resposta.get("gazettes", [])
total = resposta.get("total_gazettes", len(diarios))
print(f"📰 {total} diário(s) com '{TERMO}' no período · mostrando {len(diarios)}")
if total == 0:
    print("   Nada encontrado. Confira o termo, o período e se o município está coberto pelo projeto.")

# ── 3. Cada diário ────────────────────────────────────────────────────────
for diario in diarios:
    extra = " (edição extra)" if diario.get("is_extra_edition") else ""
    print(f"\n🗓️  {diario.get('date')} · {diario.get('territory_name')}/{diario.get('state_code')}"
          f" · edição {diario.get('edition') or '?'}{extra}")
    print(f"   📄 {diario.get('url')}")
    for trecho in diario.get("excerpts", []):
        # Os trechos vêm com quebras de linha do PDF; juntamos tudo numa linha só.
        print(f"   💬 {' '.join(trecho.split())[:250]}")

# ── 4. Paginação ──────────────────────────────────────────────────────────
if total > len(diarios):
    print(f"\n➡️  Há mais {total - len(diarios)} resultado(s). Peça a próxima página com offset={len(diarios)}.")

### `br/02_querido_diario_csv.py` — Vários termos, vários municípios: guardar os achados do Querido Diário num CSV.

In [ ]:
%cd -q {PASTA_NB}
"""
Vários termos, vários municípios: guardar os achados do Querido Diário num CSV.

Carta 30 (rank B) — Jutsu do Diário Oficial (parte 2)

O que você aprende:
  • a repetir a mesma consulta para uma lista de termos × municípios (laço duplo);
  • a fazer uma pausa entre pedidos (educação com a API pública);
  • a gravar os resultados num CSV com a data da coleta (o "ddg_search_to_csv" do original, em versão BR);
  • a tirar resultados repetidos (o mesmo diário pode aparecer para dois termos).

Como rodar (dentro da pasta dia-09-resultados-de-busca):
    python br/02_querido_diario_csv.py
    OSINT_OFFLINE=1 python br/02_querido_diario_csv.py
"""

import csv
import json
import os
import time
import urllib.parse
import urllib.request
from datetime import datetime, timedelta, timezone
from pathlib import Path

# ── Parâmetros ────────────────────────────────────────────────────────────
TERMOS = "PIX, golpe, comércio eletrônico"  #@param {type:"string"}
# Códigos IBGE separados por vírgula: Belo Horizonte, Contagem, Betim
MUNICIPIOS = "3106200, 3118601, 3106705"  #@param {type:"string"}
DESDE = "2025-10-01"  #@param {type:"date"}
POR_CONSULTA = 10  #@param {type:"integer"}
PAUSA_SEGUNDOS = 1.5  #@param {type:"number"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
OFFLINE = os.environ.get("OSINT_OFFLINE") == "1"
SAIDAS = PASTA_DIA / "saidas"

try:
    from zoneinfo import ZoneInfo
    FUSO_BR = ZoneInfo("America/Sao_Paulo")
except Exception:
    FUSO_BR = timezone(timedelta(hours=-3))


def consultar(termo, codigo):
    """Uma consulta ao Querido Diário. Devolve a lista 'gazettes' (ou a amostra)."""
    parametros = {"querystring": termo, "territory_ids": codigo, "published_since": DESDE,
                  "size": POR_CONSULTA, "excerpt_size": 200, "number_of_excerpts": 1}
    url = "https://api.queridodiario.ok.org.br/gazettes?" + urllib.parse.urlencode(parametros)
    pedido = urllib.request.Request(url, headers={"User-Agent": "python-para-osint-21-dias/estudo"})
    with urllib.request.urlopen(pedido, timeout=30) as resposta:
        return json.load(resposta).get("gazettes", [])


termos = [t.strip() for t in TERMOS.split(",") if t.strip()]
codigos = [c.strip() for c in MUNICIPIOS.split(",") if c.strip()]
coletado_em = datetime.now(FUSO_BR).isoformat(timespec="seconds")

# ── 1. Coletar ────────────────────────────────────────────────────────────
linhas = {}             # chave = (url, termo) → evita duplicar a mesma linha
usar_amostra = OFFLINE
if OFFLINE:
    print("⚠️ MODO OFFLINE: usando a amostra local.")

for termo in termos:
    for codigo in codigos:
        diarios = None
        if not usar_amostra:
            try:
                diarios = consultar(termo, codigo)
                time.sleep(PAUSA_SEGUNDOS)          # não sobrecarregar um serviço público e gratuito
            except Exception as erro:
                print(f"⚠️ MODO OFFLINE: a consulta falhou ({erro}). Usando a amostra local.")
                usar_amostra = True                 # caiu uma vez? usamos a amostra no resto
        if diarios is None:
            amostra = json.loads((PASTA_DIA / "dados" / "querido_diario_bh.json").read_text(encoding="utf-8"))
            # A amostra só tem BH; filtramos pelo município e pelo termo para simular a busca.
            diarios = [g for g in amostra["gazettes"] if g.get("territory_id") == codigo
                       and any(termo.casefold() in e.casefold() for e in g.get("excerpts", []))]
        print(f"🔎 {termo!r:<24} em {codigo}: {len(diarios)} diário(s)")
        for g in diarios:
            linhas[(g.get("url"), termo)] = {
                "termo": termo,
                "codigo_ibge": g.get("territory_id", codigo),
                "municipio": g.get("territory_name", ""),
                "uf": g.get("state_code", ""),
                "data": g.get("date", ""),
                "edicao": g.get("edition") or "",
                "extra": "sim" if g.get("is_extra_edition") else "não",
                "trecho": " ".join(" ".join(g.get("excerpts", [])).split())[:300],
                "url": g.get("url", ""),
                "coletado_em": coletado_em,          # quando NÓS vimos (importante para a prova)
            }

# ── 2. Gravar ─────────────────────────────────────────────────────────────
SAIDAS.mkdir(exist_ok=True)
destino = SAIDAS / "querido_diario_resultados.csv"
campos = ["termo", "codigo_ibge", "municipio", "uf", "data", "edicao", "extra", "trecho", "url", "coletado_em"]
with open(destino, "w", encoding="utf-8-sig", newline="") as arquivo:
    escritor = csv.DictWriter(arquivo, fieldnames=campos, delimiter=";")
    escritor.writeheader()
    # Ordenamos por data (mais recente primeiro) para facilitar a leitura.
    escritor.writerows(sorted(linhas.values(), key=lambda l: l["data"], reverse=True))
print(f"\n💾 {len(linhas)} linha(s) em {destino.relative_to(PASTA_DIA)}")

### `br/03_dorks_br.py` — Gerador de dorks: montar buscas avançadas e abrir com um clique (sem raspar buscador).

In [ ]:
%cd -q {PASTA_NB}
"""
Gerador de dorks: montar buscas avançadas e abrir com um clique (sem raspar buscador).

Carta 30 (rank B) — Jutsu do Diário Oficial (parte 3)

O que você aprende:
  • os operadores de busca mais úteis: site:, filetype:, intext:, intitle:, aspas, -excluir, OR;
  • a montar a URL de busca do Google, DuckDuckGo e Bing com urllib.parse.quote_plus;
  • a gerar uma página HTML com os links (html.escape para não quebrar o HTML);
  • por que NÃO raspamos o buscador: termos de uso, bloqueios e captchas. Você clica, você lê.

Como rodar (dentro da pasta dia-09-resultados-de-busca):
    python br/03_dorks_br.py
    (depois abra saidas/dorks.html no navegador)
"""

import html                 # escapar textos antes de colocar no HTML
import urllib.parse         # transformar a busca em texto seguro para URL
from datetime import datetime, timedelta, timezone
from pathlib import Path

# ── Parâmetros: termos do caso (FICTÍCIOS) e um alvo institucional ────────
DOMINIO_LOJA = "megaoferta-brasil.example"  #@param {type:"string"}
APELIDO = "kau.ofertas"  #@param {type:"string"}
CNPJ_RODAPE = "48.217.093/0001-30"  #@param {type:"string"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
SAIDAS = PASTA_DIA / "saidas"

try:
    from zoneinfo import ZoneInfo
    FUSO_BR = ZoneInfo("America/Sao_Paulo")
except Exception:
    FUSO_BR = timezone(timedelta(hours=-3))

# ── 1. O "cardápio" de buscas (grupo, objetivo, consulta) ─────────────────
DORKS = [
    ("Treino com fonte oficial", "PDFs do governo que citam CNPJ",
     'site:gov.br filetype:pdf intext:"CNPJ"'),
    ("Treino com fonte oficial", "Alertas oficiais sobre golpes com PIX",
     'site:gov.br intitle:"golpe" "PIX"'),
    ("Treino com fonte oficial", "Planilhas públicas de contratos em MG",
     'site:mg.gov.br (filetype:xlsx OR filetype:csv) "contrato"'),
    ("Caso: a loja", "Páginas que citam o domínio (fora do próprio site)",
     f'"{DOMINIO_LOJA}" -site:{DOMINIO_LOJA}'),
    ("Caso: a loja", "Reclamações de consumidores",
     f'"megaoferta brasil" (golpe OR reclamação OR "não entregou")'),
    ("Caso: a loja", "O CNPJ inventado em outros sites (reaproveitado?)",
     f'"{CNPJ_RODAPE}"'),
    ("Caso: o vendedor", "O apelido em redes e fóruns",
     f'"{APELIDO}"'),
    ("Caso: o vendedor", "Apelido + WhatsApp/Telegram",
     f'"{APELIDO}" (whatsapp OR telegram OR "t.me")'),
]

# ── 2. Cada buscador tem seu endereço de busca ────────────────────────────
BUSCADORES = {
    "Google": "https://www.google.com/search?q=",
    "DuckDuckGo": "https://duckduckgo.com/?q=",
    "Bing": "https://www.bing.com/search?q=",
}


def link(base, consulta):
    """quote_plus troca espaço por '+' e codifica aspas, dois-pontos, acentos…"""
    return base + urllib.parse.quote_plus(consulta)


# Mostrar no terminal um exemplo de "antes e depois" da codificação
exemplo = DORKS[0][2]
print(f"🔤 Consulta:  {exemplo}")
print(f"🔗 Na URL:    {link(BUSCADORES['Google'], exemplo)}\n")

# ── 3. Montar o HTML ──────────────────────────────────────────────────────
agora = datetime.now(FUSO_BR).strftime("%d/%m/%Y %H:%M")
partes = [
    "<!doctype html><html lang='pt-BR'><head><meta charset='utf-8'>",
    "<title>Dorks do caso Vitrine Fantasma</title>",
    "<style>body{font-family:system-ui,sans-serif;max-width:900px;margin:2rem auto;padding:0 1rem}"
    "code{background:#f1f1f1;padding:2px 6px;border-radius:4px}td,th{padding:6px;border-bottom:1px solid #ddd;"
    "text-align:left;vertical-align:top}table{border-collapse:collapse;width:100%}</style></head><body>",
    "<h1>🔎 Dorks — Operação Vitrine Fantasma</h1>",
    f"<p>Gerado em {agora} (horário de Brasília). Clique, leia e <b>registre</b> o que achar "
    "(URL + data/hora + captura de tela).</p>",
]
grupo_atual = None
for grupo, objetivo, consulta in DORKS:
    if grupo != grupo_atual:                          # novo grupo → nova tabela
        if grupo_atual is not None:
            partes.append("</table>")
        partes.append(f"<h2>{html.escape(grupo)}</h2><table><tr><th>Objetivo</th><th>Consulta</th><th>Abrir</th></tr>")
        grupo_atual = grupo
    botoes = " · ".join(
        f"<a href='{html.escape(link(base, consulta), quote=True)}' target='_blank' rel='noopener noreferrer'>{nome}</a>"
        for nome, base in BUSCADORES.items())
    partes.append(f"<tr><td>{html.escape(objetivo)}</td><td><code>{html.escape(consulta)}</code></td><td>{botoes}</td></tr>")
partes.append("</table><p><small>Os dados do caso são fictícios: as buscas do caso devem voltar vazias. "
              "Use as de treino para ver os operadores funcionando.</small></p></body></html>")

SAIDAS.mkdir(exist_ok=True)
destino = SAIDAS / "dorks.html"
destino.write_text("\n".join(partes), encoding="utf-8")

# ── 4. Resumo no terminal ─────────────────────────────────────────────────
for grupo, objetivo, consulta in DORKS:
    print(f"   [{grupo}] {objetivo}\n      {consulta}")
print(f"\n💾 {len(DORKS)} dorks × {len(BUSCADORES)} buscadores em {destino.relative_to(PASTA_DIA)}")
print("   Abra no navegador. Nada foi buscado automaticamente: quem clica é você.")

## ⚖️ Ética, LGPD e OPSEC

- **Não raspe buscadores.** Os termos de uso do Google e do Bing proíbem consultas automatizadas fora
  das APIs oficiais. Além de bloqueio e captcha, você "suja" o IP da sua organização. Gerar os links e
  clicar é legítimo; robô metralhando o buscador, não.
- **Dork é lupa, não pé de cabra.** Achar um arquivo exposto não autoriza baixá-lo, ainda mais se tiver
  dados pessoais (reveja o Dia 8). Use dorks para entender exposição e para achar fontes públicas.
- **Suas buscas contam uma história.** O buscador registra o que você procura, logado ou não. Buscar o
  nome de um suspeito na sua conta pessoal mistura sua identidade com a investigação. Use o ambiente
  de trabalho e a conta institucional (mais no Dia 12).
- **Registre data e hora da coleta.** Resultado de busca muda todo dia. O que não foi registrado
  (URL + data/hora + captura) não existe para o relatório.
- **Diário Oficial é público, mas a finalidade importa.** Nomes de servidores e cidadãos publicados
  continuam protegidos pela LGPD quando reutilizados: use só o que tiver relação com o caso.

## 🎯 Quiz rápido

Escolha a resposta no formulário de cada célula e rode a célula.

In [ ]:
#@title 1. Por que o original ddg_search.py não roda mais em 2026?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) O DuckDuckGo saiu do ar", "B) A função ddg() foi removida; hoje o pacote se chama ddgs e usa DDGS().text()", "C) O Python 3 não aceita mais from ... import", "D) Busca na internet passou a ser proibida"]
correta = "B) A função ddg() foi removida; hoje o pacote se chama ddgs e usa DDGS().text()"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "O pacote mudou de interface (e de nome). Por isso o original fica intacto e a versão 2026 fica ao lado.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 2. O que a busca site:gov.br filetype:pdf intext:'CNPJ' pede ao buscador?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Sites que vendem PDFs de CNPJ", "B) PDFs hospedados em domínios gov.br que contenham a palavra CNPJ no texto", "C) O CNPJ do site gov.br", "D) Todos os PDFs da internet"]
correta = "B) PDFs hospedados em domínios gov.br que contenham a palavra CNPJ no texto"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "site: restringe o domínio, filetype: o tipo de arquivo e intext: exige o termo no conteúdo.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 3. Na resposta do Querido Diário, qual a diferença entre total_gazettes e len(gazettes)?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Nenhuma, são sempre iguais", "B) total_gazettes é quantos diários existem para a busca; gazettes traz só a página pedida (size)", "C) total_gazettes conta palavras", "D) gazettes inclui diários de outros municípios"]
correta = "B) total_gazettes é quantos diários existem para a busca; gazettes traz só a página pedida (size)"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "A API é paginada: para ver o resto, peça de novo com offset.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

# 🏆 Desafio S — Dia 9: Vigia do Diário Oficial

**Carta 51 · Rank S**

## A missão

A Iara não quer olhar o Diário Oficial todo dia à mão. Ela quer um **vigia**: um script que roda de
manhã, consulta o Querido Diário e mostra **só o que é novo** desde a última vez. Se a prefeitura
publicar um alerta sobre lojas falsas, uma sanção a uma empresa ou um termo novo ligado ao caso, o NFA
fica sabendo no mesmo dia.

Escreva um script que:

1. Receba uma lista de termos (`TERMOS`) e um código IBGE de município (`CODIGO_IBGE`).
2. Consulte o Querido Diário para cada termo, nos últimos `DIAS_PARA_TRAS` dias, com pausa entre pedidos.
3. Guarde em `saidas/vigia_estado.json` o que já foi visto (uma "impressão digital" por diário + termo).
4. Na execução seguinte, mostre **apenas as publicações novas**.
5. Gere `saidas/vigia_novidades.html` com data, termo, trecho e link (tudo com `html.escape`).
6. Funcione **sem internet** com a amostra de `dados/`.

## Critérios de rank S

- [ ] Só biblioteca padrão
- [ ] Rodando duas vezes seguidas, a segunda mostra **zero** novidades
- [ ] Se o arquivo de estado estiver apagado ou corrompido, o script recomeça sem quebrar
- [ ] O HTML abre no navegador e nenhum trecho "quebra" a página

## Dica

Um `set` de textos (`f"{url}|{termo}"`) responde "já vi isso?" instantaneamente. Para salvar em JSON,
converta para lista ordenada (`sorted(vistos)`), como no Dia 6.

In [ ]:
%cd -q {PASTA_NB}
# ✍️ Escreva sua solução aqui.

In [ ]:
#@title 🔒 Gabarito (dê dois cliques aqui para ver o código) { display-mode: "form" }
%cd -q {PASTA_NB}
"""
Gabarito do Desafio S do Dia 9 — Vigia do Diário Oficial.

Consulta o Querido Diário para uma lista de termos em um município, compara com
o que já foi visto nas execuções anteriores (saidas/vigia_estado.json) e mostra
SÓ as publicações novas. Grava um relatório HTML com links e trechos.
Sem internet (ou com OSINT_OFFLINE=1), usa a amostra local.
"""

import html
import json
import os
import time
import urllib.parse
import urllib.request
from datetime import datetime, timedelta, timezone
from pathlib import Path

TERMOS = "PIX, Black Friday, consumidor"  #@param {type:"string"}
CODIGO_IBGE = "3106200"  #@param {type:"string"}
DIAS_PARA_TRAS = 120  #@param {type:"integer"}

PASTA_DIA = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
SAIDAS = PASTA_DIA / "saidas"
ESTADO = SAIDAS / "vigia_estado.json"
OFFLINE = os.environ.get("OSINT_OFFLINE") == "1"

try:
    from zoneinfo import ZoneInfo
    FUSO_BR = ZoneInfo("America/Sao_Paulo")
except Exception:
    FUSO_BR = timezone(timedelta(hours=-3))

agora = datetime.now(FUSO_BR)
desde = (agora - timedelta(days=DIAS_PARA_TRAS)).date().isoformat()


def buscar(termo):
    if not OFFLINE:
        try:
            q = urllib.parse.urlencode({"querystring": termo, "territory_ids": CODIGO_IBGE, "published_since": desde,
                                        "size": 20, "excerpt_size": 200, "number_of_excerpts": 1})
            req = urllib.request.Request("https://api.queridodiario.ok.org.br/gazettes?" + q,
                                         headers={"User-Agent": "python-para-osint-21-dias/estudo"})
            with urllib.request.urlopen(req, timeout=30) as resposta:
                dados = json.load(resposta)
            time.sleep(1)
            return dados.get("gazettes", [])
        except Exception as erro:
            print(f"⚠️ MODO OFFLINE: a consulta falhou ({erro}). Usando a amostra local.")
    amostra = json.loads((PASTA_DIA / "dados" / "querido_diario_bh.json").read_text(encoding="utf-8"))
    # A amostra é de 2025: no modo offline ignoramos o filtro de data e filtramos só pelo termo.
    return [g for g in amostra["gazettes"] if g.get("territory_id") == CODIGO_IBGE
            and any(termo.casefold() in e.casefold() for e in g.get("excerpts", []))]


if OFFLINE:
    print("⚠️ MODO OFFLINE: usando a amostra local.")

# Estado anterior: conjunto de "impressões digitais" (url + termo) já vistas
try:
    vistos = set(json.loads(ESTADO.read_text(encoding="utf-8"))["vistos"])
except (FileNotFoundError, KeyError, json.JSONDecodeError):
    vistos = set()
primeira_vez = not vistos

novidades = []
for termo in [t.strip() for t in TERMOS.split(",") if t.strip()]:
    for g in buscar(termo):
        chave = f"{g.get('url')}|{termo}"
        if chave not in vistos:
            vistos.add(chave)
            novidades.append({"termo": termo, "data": g.get("date", ""), "url": g.get("url", ""),
                              "municipio": g.get("territory_name", ""),
                              "trecho": " ".join(" ".join(g.get("excerpts", [])).split())[:300]})

novidades.sort(key=lambda n: n["data"], reverse=True)
print(f"{'🆕 Primeira execução' if primeira_vez else '🔁 Execução de acompanhamento'}: "
      f"{len(novidades)} publicação(ões) nova(s).")
for n in novidades:
    print(f"   {n['data']}  [{n['termo']}]  {n['url']}")

SAIDAS.mkdir(exist_ok=True)
ESTADO.write_text(json.dumps({"atualizado_em": agora.isoformat(timespec="seconds"),
                              "vistos": sorted(vistos)}, indent=2, ensure_ascii=False), encoding="utf-8")
itens = "".join(
    f"<li><b>{html.escape(n['data'])}</b> · {html.escape(n['termo'])} · {html.escape(n['municipio'])}<br>"
    f"<small>{html.escape(n['trecho'])}</small><br>"
    f"<a href='{html.escape(n['url'], quote=True)}' target='_blank' rel='noopener noreferrer'>abrir diário</a></li>"
    for n in novidades) or "<li>Nenhuma novidade desde a última execução.</li>"
(SAIDAS / "vigia_novidades.html").write_text(
    f"<!doctype html><html lang='pt-BR'><meta charset='utf-8'><title>Vigia do Diário Oficial</title>"
    f"<body style='font-family:sans-serif;max-width:800px;margin:auto'><h1>Vigia do Diário Oficial</h1>"
    f"<p>Município {html.escape(CODIGO_IBGE)} · gerado em {agora:%d/%m/%Y %H:%M}</p><ul>{itens}</ul></body></html>",
    encoding="utf-8")
print("💾 saidas/vigia_estado.json e saidas/vigia_novidades.html gravados. Rode de novo: zero novidades.")

## 🔎 Inovação OSINT do dia

**Querido Diário (Open Knowledge Brasil).** Os Diários Oficiais municipais costumam ser PDFs espalhados
em milhares de sites diferentes, cada um com seu formato. O Querido Diário raspa, converte para texto e
oferece uma **API pública e gratuita** de busca por termo, município (código IBGE) e período, com links
para o PDF original. Para OSINT brasileiro, é uma das fontes oficiais mais subestimadas: contratos,
nomeações, sanções, alvarás e alertas ao consumidor, tudo pesquisável. Biblioteca Python simples:
`urllib.request` + `urllib.parse` + `json` (nativos).

## 📚 Para ir além

- Querido Diário (busca no navegador): <https://queridodiario.ok.org.br/>
- Documentação interativa da API: <https://api.queridodiario.ok.org.br/docs>
- Pacote `ddgs` no PyPI: <https://pypi.org/project/ddgs/>
- Google Hacking Database (Exploit-DB): <https://www.exploit-db.com/google-hacking-database>
- Lista de operadores de busca do autor: <https://github.com/cipher387/Advanced-search-operators-list>

## ✅ Checklist

- [ ] Entendi por que os originais quebraram e rodei a versão 2026 com `ddgs`
- [ ] Busquei um termo no Querido Diário e abri o PDF de um resultado
- [ ] Salvei resultados de busca em CSV com a data da coleta
- [ ] Gerei e abri o `dorks.html`
- [ ] Fiz o desafio S
- [ ] Anotei no meu [`DIARIO.md`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/DIARIO.md)

---

🎉 Terminou? [Registre sua entrega do Dia 9](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/issues/new?template=entreguei-o-dia.yml) · Baseado em [cipher387/python-for-OSINT-21-days](https://github.com/cipher387/python-for-OSINT-21-days) (MIT) · Adaptação pt-BR: [Ridd1kulusC0d3r/python-para-osint-21-dias](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias)